# UTS Intelegensi Buatan - Kasus 1: Klasifikasi Kultivar Anggur
Nama: M.Fathir Putra Hermansyah  |  NIM: 23552011084  |  Kelas: TIF RP 23D

Alur: (1) siapkan folder & dataset, (2) tulis `fitur.py`, `latih.py`, `app.py`, (3) latih model, (4) jalankan Streamlit di Colab.

In [ ]:
!mkdir -p dataset models
!pip -q install streamlit

## 1. Upload dataset
Upload `kasus01_wine.csv` saat diminta (disimpan ke folder `dataset/`).

In [ ]:
from google.colab import files
import shutil
up = files.upload()
for nama in up:
    shutil.move(nama, 'dataset/kasus01_wine.csv')

## 2. Feature engineering (`fitur.py`)
Audit data + transformer `RasioFitur` (stateless, aman dari kebocoran data).

In [ ]:
%%writefile fitur.py
"""fitur.py - audit data dan feature engineering untuk Kasus 1 (Wine)."""
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

TARGET = "kultivar"
FITUR_ASLI = [
    "alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium",
    "total_phenols", "flavanoids", "nonflavanoid_phenols", "proanthocyanins",
    "color_intensity", "hue", "od280_od315_of_diluted_wines", "proline",
]
# Fitur rasio: pasangan fitur yang saling terkait secara kimia / korelasi tinggi
RASIO = {
    "rasio_flavanoid_fenol": ("flavanoids", "total_phenols"),            # r=0.86
    "rasio_nonflav_flav": ("nonflavanoid_phenols", "flavanoids"),
    "rasio_od280_flavanoid": ("od280_od315_of_diluted_wines", "flavanoids"),  # r=0.79
    "rasio_proline_alkohol": ("proline", "alcohol"),                      # r=0.64
    "rasio_warna_hue": ("color_intensity", "hue"),
}
# Fitur asli yang dibuang pada varian "rasio_ringkas" (redundan dgn rasio)
ASLI_DIBUANG = ["total_phenols", "od280_od315_of_diluted_wines"]
EPS = 1e-6


def muat_data(path="dataset/kasus01_wine.csv"):
    return pd.read_csv(Path(path))


def audit_data(df):
    """Audit: bentuk, tipe, missing, duplikat, nilai tidak wajar, korelasi tinggi."""
    X = df[FITUR_ASLI]
    korelasi = X.corr().abs().where(np.triu(np.ones((13, 13), bool), 1)).stack()
    return {
        "bentuk": df.shape,
        "tipe": df.dtypes.astype(str).to_dict(),
        "missing_total": int(df.isna().sum().sum()),
        "duplikat": int(df.duplicated().sum()),
        "nilai_tak_wajar_<=0": int((X <= 0).sum().sum()),
        "distribusi_kelas": df[TARGET].value_counts().to_dict(),
        "korelasi_tinggi(>0.6)": korelasi[korelasi > 0.6].sort_values(ascending=False).round(3).to_dict(),
    }


class RasioFitur(BaseEstimator, TransformerMixin):
    """Menambah fitur rasio. Tidak punya statistik yang dipelajari (stateless),
    jadi aman dari kebocoran data. Input & output: DataFrame.

    mode:
      'asli'         -> 13 fitur asli saja
      'asli+rasio'   -> 13 asli + 5 rasio
      'rasio_ringkas'-> 5 rasio + asli, tapi total_phenols & od280 dibuang
    """

    def __init__(self, mode="asli+rasio"):
        self.mode = mode

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if self.mode == "asli":
            return X[FITUR_ASLI]
        for nama, (a, b) in RASIO.items():
            X[nama] = X[a] / (X[b] + EPS)
        if self.mode == "rasio_ringkas":
            X = X.drop(columns=ASLI_DIBUANG)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.transform(pd.DataFrame(
            [[1.0] * 13], columns=FITUR_ASLI)).columns)


## 3. Pelatihan (`latih.py`)
Split train/test lebih dulu, pilih model dengan cross-validation di train, evaluasi test satu kali, simpan ke `models/`.

In [ ]:
%%writefile latih.py
"""latih.py - pelatihan, pemilihan model (CV di train), evaluasi test 1x, simpan joblib."""
import json
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import (RepeatedStratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from fitur import FITUR_ASLI, TARGET, RasioFitur, audit_data, muat_data

# Sembunyikan FutureWarning SVC(probability=True) di proses worker CV
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

SEED = 42
DIR_MODEL = Path("models")


def buat_pipeline(mode, estimator):
    # Scaler berada DI DALAM pipeline -> di-fit hanya pada data train (per fold saat CV)
    return Pipeline([("fitur", RasioFitur(mode=mode)),
                     ("scaler", StandardScaler()),
                     ("model", estimator)])


def kandidat_model():
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
        "SVM (RBF)": SVC(probability=True, random_state=SEED),
    }


def main():
    DIR_MODEL.mkdir(exist_ok=True)
    df = muat_data()
    print("== AUDIT DATA ==")
    for k, v in audit_data(df).items():
        print(f"{k}: {v}")

    X, y = df[FITUR_ASLI], df[TARGET]
    # 1) SPLIT DULU, sebelum scaling/rasio/apa pun
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED)
    print(f"\nTrain={len(X_tr)}  Test={len(X_te)}")

    # 2) Pemilihan model & varian fitur: CV hanya di data train
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    hasil = []
    for mode in ["asli", "asli+rasio", "rasio_ringkas"]:
        for nama, est in kandidat_model().items():
            skor = cross_val_score(buat_pipeline(mode, est), X_tr, y_tr,
                                   cv=cv, scoring="f1_macro", n_jobs=-1)
            hasil.append({"fitur": mode, "model": nama,
                          "cv_f1_macro_mean": skor.mean(), "cv_f1_macro_std": skor.std()})
    tabel = pd.DataFrame(hasil).sort_values("cv_f1_macro_mean", ascending=False)
    print("\n== HASIL CV (f1_macro, data train) ==")
    print(tabel.round(4).to_string(index=False))

    terbaik = tabel.iloc[0]
    # Jika selisih <= 0.005 dengan varian 'asli', pilih yang lebih sederhana
    kandidat_sederhana = tabel[(tabel.fitur == "asli") & (tabel.model == terbaik.model)].iloc[0]
    if terbaik.cv_f1_macro_mean - kandidat_sederhana.cv_f1_macro_mean <= 0.005:
        terbaik = kandidat_sederhana
    print(f"\nTerpilih: {terbaik.model} | fitur={terbaik.fitur}")

    # 3) Fit final di seluruh train, evaluasi test SATU kali
    pipe = buat_pipeline(terbaik.fitur, kandidat_model()[terbaik.model]).fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    kelas = list(pipe.classes_)
    metrik = {
        "accuracy": accuracy_score(y_te, pred),
        "f1_macro": f1_score(y_te, pred, average="macro"),
        "precision_macro": precision_score(y_te, pred, average="macro"),
        "recall_macro": recall_score(y_te, pred, average="macro"),
    }
    cm = confusion_matrix(y_te, pred, labels=kelas)
    print("\n== EVALUASI TEST (1x) ==")
    print({k: round(v, 4) for k, v in metrik.items()})
    print(pd.DataFrame(cm, index=kelas, columns=kelas))
    print(classification_report(y_te, pred, digits=3))

    # 4) Simpan pipeline + metadata untuk Streamlit
    metadata = {
        "fitur_input": FITUR_ASLI,
        "kelas": kelas,
        "model_terpilih": terbaik.model,
        "mode_fitur": terbaik.fitur,
        "rentang_fitur": {c: [float(X_tr[c].min()), float(X_tr[c].max()),
                              float(X_tr[c].median())] for c in FITUR_ASLI},
        "metrik_test": metrik,
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(y_te, pred, output_dict=True),
        "hasil_cv": tabel.to_dict(orient="records"),
        "n_train": len(X_tr), "n_test": len(X_te),
    }
    joblib.dump({"pipeline": pipe, "metadata": metadata}, DIR_MODEL / "wine_pipeline.joblib")
    (DIR_MODEL / "metadata.json").write_text(json.dumps(metadata, indent=2))
    print("\nTersimpan: models/wine_pipeline.joblib, models/metadata.json")


if __name__ == "__main__":
    main()


In [ ]:
!python latih.py

## 4. Aplikasi Streamlit (`app.py`)

In [ ]:
%%writefile app.py
"""app.py - aplikasi Streamlit klasifikasi kultivar anggur (Kasus 1)."""
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

import fitur  # noqa: F401  (diperlukan joblib untuk memuat RasioFitur)

PATH_MODEL = Path("models") / "wine_pipeline.joblib"
LABEL = {
    "alcohol": "Alkohol", "malic_acid": "Asam malat", "ash": "Abu",
    "alcalinity_of_ash": "Alkalinitas abu", "magnesium": "Magnesium",
    "total_phenols": "Total fenol", "flavanoids": "Flavanoid",
    "nonflavanoid_phenols": "Fenol non-flavanoid",
    "proanthocyanins": "Proantosianin", "color_intensity": "Intensitas warna",
    "hue": "Hue", "od280_od315_of_diluted_wines": "OD280/OD315",
    "proline": "Prolin",
}


@st.cache_resource
def muat_model():
    bundel = joblib.load(PATH_MODEL)
    return bundel["pipeline"], bundel["metadata"]


def prediksi(pipe, meta, nilai: dict):
    """Input mentah (13 fitur asli) -> (label, tabel probabilitas)."""
    X = pd.DataFrame([nilai], columns=meta["fitur_input"])
    proba = pipe.predict_proba(X)[0]
    tabel = pd.Series(proba, index=pipe.classes_, name="Probabilitas")
    return tabel.idxmax(), tabel


def main():
    st.set_page_config(page_title="Klasifikasi Kultivar Anggur", page_icon="🍷")
    st.title("🍷 Klasifikasi Kultivar Anggur")
    st.caption("Input hasil analisis kimia mentah; scaling dan fitur rasio diproses otomatis oleh pipeline.")

    if not PATH_MODEL.exists():
        st.error("Model belum ada. Jalankan `python latih.py` terlebih dahulu.")
        st.stop()
    pipe, meta = muat_model()

    tab_pred, tab_metrik = st.tabs(["Prediksi", "Metrik evaluasi"])

    with tab_pred:
        st.subheader("Masukkan hasil analisis kimia")
        nilai = {}
        kiri, kanan = st.columns(2)
        for i, kol in enumerate(meta["fitur_input"]):
            lo, hi, med = meta["rentang_fitur"][kol]
            margin = (hi - lo) * 0.25
            with (kiri if i % 2 == 0 else kanan):
                nilai[kol] = st.number_input(
                    LABEL.get(kol, kol), min_value=max(0.0, lo - margin),
                    max_value=hi + margin, value=med, format="%.3f",
                    help=f"Rentang data latih: {lo:.2f} - {hi:.2f}")

        if st.button("Prediksi", type="primary"):
            label, tabel = prediksi(pipe, meta, nilai)
            st.success(f"Prediksi: **{label}** ({tabel.max():.1%})")
            st.bar_chart(tabel)
            st.dataframe(tabel.map("{:.2%}".format).to_frame())
            diluar = [k for k, v in nilai.items()
                      if not meta["rentang_fitur"][k][0] <= v <= meta["rentang_fitur"][k][1]]
            if diluar:
                st.warning("Nilai di luar rentang data latih (prediksi kurang andal): "
                           + ", ".join(diluar))

    with tab_metrik:
        m = meta["metrik_test"]
        st.write(f"Model: **{meta['model_terpilih']}** | varian fitur: `{meta['mode_fitur']}` "
                 f"| train={meta['n_train']}, test={meta['n_test']}")
        c = st.columns(4)
        c[0].metric("Accuracy", f"{m['accuracy']:.3f}")
        c[1].metric("F1 macro", f"{m['f1_macro']:.3f}")
        c[2].metric("Precision macro", f"{m['precision_macro']:.3f}")
        c[3].metric("Recall macro", f"{m['recall_macro']:.3f}")
        st.subheader("Confusion matrix (data test)")
        st.dataframe(pd.DataFrame(meta["confusion_matrix"], index=[f"Aktual {k}" for k in meta["kelas"]],
                                  columns=[f"Prediksi {k}" for k in meta["kelas"]]))
        st.subheader("Classification report")
        rep = pd.DataFrame(meta["classification_report"]).T.round(3)
        st.dataframe(rep)
        st.subheader("Perbandingan model (cross-validation di data train)")
        st.dataframe(pd.DataFrame(meta["hasil_cv"]).round(4))


main()


## 5. Jalankan Streamlit di Colab
Setelah sel di bawah jalan, salin angka **IP** yang tercetak (password tunnel), lalu buka link `https://...loca.lt` dan tempel IP tersebut.

In [ ]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run app.py --server.port 8501 &>/content/log.txt &
!npx -y localtunnel --port 8501